In [105]:
import torch
import torch.nn as nn
import pydiffvg
import torch.nn.functional as F
import numpy as np
import torchvision.transforms as T
from PIL import Image
import pandas as pd
import os

def get_premium_colors_from_t(t_values, cmap_name='magma', device='cuda'):
    """
    Samples colors from a colormap using specific 't' values (0.0 to 1.0).
    t_values: Tensor of shape [N]
    """
    # 1. Get Colormap
    cmap = matplotlib.cm.get_cmap(cmap_name)
    
    # 2. Sample using the provided t_values (Arc Length)
    # Convert tensor to numpy for matplotlib
    t_numpy = t_values.detach().cpu().numpy()
    
    # 3. Get RGBA
    colors_numpy = cmap(t_numpy)
    
    # 4. Convert back to Tensor
    return torch.from_numpy(colors_numpy).to(device).float()

# Ensure output directory exists
if not os.path.exists("fill_steps"):
    os.makedirs("fill_steps")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
canvas_width, canvas_height = 512, 512

class UnitaryBSpline3D_New(nn.Module):
    def __init__(self, num_kp, kp_init, device):
        super().__init__()
        self.num_kp = num_kp
        self.device = device
        
        # Quintic Matrix (Keep as buffer)
        self.register_buffer('M_B', torch.tensor([
            [1/120, 26/120, 66/120, 26/120, 1/120, 0],
            [0, 10/120, 60/120, 40/120, 10/120, 0],
            [0, 0, 40/120, 60/120, 20/120, 0],
            [0, 0, 20/120, 60/120, 40/120, 0],
            [0, 0, 10/120, 40/120, 60/120, 10/120],
            [0, 1/120, 26/120, 66/120, 26/120, 1/120]
        ], dtype=torch.float32, device=device))

        # 1. PARAMETERS (Optimized in 3D Space)
        # kp_init should be [num_kp, 3]
        self.points_3d = nn.Parameter(kp_init.clone().detach().to(device))
        self.unit_widths = nn.Parameter(torch.full((num_kp, 1), 0.3, device=device))
        
        # Jerk Loss Matrix
        self.register_buffer('G', self.get_gram_matrix_jerk(num_kp, device))
    

    @torch.no_grad()
    def set_data(self, new_points, new_widths):
        """Safely updates the geometry and recalculates all dependent matrices (G)."""
        device = self.device
        
        if new_points.dim() == 1: new_points = new_points.unsqueeze(0)
        if new_widths.dim() == 1: new_widths = new_widths.unsqueeze(1)
        
        if new_points.shape[0] != new_widths.shape[0]:
            raise ValueError(f"Size Mismatch: {new_points.shape[0]} points vs {new_widths.shape[0]} widths")

        self.num_kp = new_points.shape[0]

        self.points_3d = nn.Parameter(new_points.contiguous().to(device))
        self.unit_widths = nn.Parameter(new_widths.contiguous().to(device))

        new_G = self.get_gram_matrix_jerk(self.num_kp, device)
        self.register_buffer('G', new_G)


    def project_points(self, points_3d, mvp_matrix, width, height):
        """Projects 3D points to 2D screen space with Z-depth."""
        pts_homo = torch.cat([points_3d, torch.ones(points_3d.shape[0], 1, device=self.device)], dim=1)
        pts_clip = pts_homo @ mvp_matrix.T
        w = pts_clip[:, 3:4]
        pts_ndc = pts_clip[:, :2] / (w + 1e-6)
        
        screen_x = (pts_ndc[:, 0] + 1) * (width / 2)
        screen_y = (1 - pts_ndc[:, 1]) * (height / 2)
        return torch.stack([screen_x, screen_y], dim=1), w.squeeze()

    def to_bezier(self, keypoints):
        """Vectorized B-Spline to Bezier conversion."""
        # Padding for quintic (pass 2 points on each end)
        k = torch.cat([keypoints[0:1].repeat(2,1), keypoints, keypoints[-1:].repeat(2,1)], dim=0)
        segments = k.unfold(0, 6, 1) # [num_segments, dim, 6]
        bezier_segments = torch.matmul(segments, self.M_B.T)
        
        main_body = bezier_segments[:-1, :, :3]
        last_seg = bezier_segments[-1:, :, :]
        
        return torch.cat([
            main_body.transpose(1, 2).reshape(-1, keypoints.shape[-1]), 
            last_seg.transpose(1, 2).reshape(-1, keypoints.shape[-1])
        ], dim=0)
    
    def get_dynamic_normals(self, points_3d, mesh_verts, mesh_normals):
        """Finds the surface normal for each point by querying the nearest mesh vertex."""
        distances = torch.cdist(points_3d, mesh_verts) 
        nearest_indices = torch.argmin(distances, dim=1)
        dynamic_normals = mesh_normals[nearest_indices]
        return dynamic_normals

    def forward(self, camera, view_w, view_h, mesh_verts, mesh_normals, mesh_depth_map, ref_dist=5.0, w_min=0.2, w_max=8.5):
        mvp_matrix = camera.get_full_matrix()
        
        # ---------------------------------------------------------
        # 1. 3D B-Spline -> 3D Bezier
        # ---------------------------------------------------------
        # Convert control points and widths to Bezier space FIRST.
        # This preserves exact 3D geometry before perspective distortion.
        bezier_3d = self.to_bezier(self.points_3d)
        base_widths = self.to_bezier(self.unit_widths).squeeze()
        
        # ---------------------------------------------------------
        # 2. 3D Bezier -> 2D Bezier (Projection)
        # ---------------------------------------------------------
        # Project the 3D Bezier control points directly to 2D screen space
        bezier_2d, bezier_depths = self.project_points(
            bezier_3d, mvp_matrix, view_w, view_h
        )
        
        # ---------------------------------------------------------
        # 3. GLOBAL OCCLUSION CULLING (Depth Map Lookup)
        # ---------------------------------------------------------
        # Convert bezier_2d (pixel coordinates) to normalized [-1, 1] coordinates
        x_norm = (bezier_2d[:, 0] / (view_w / 2.0)) - 1.0
        y_norm = (bezier_2d[:, 1] / (view_h / 2.0)) - 1.0 
        
        grid = torch.stack([x_norm, y_norm], dim=-1).view(1, 1, -1, 2)
        
        # Sample the depth map at the projected Bezier point coordinates
        sampled_mesh_depths = F.grid_sample(
            mesh_depth_map, grid, mode='bilinear', padding_mode='border', align_corners=False
        ).view(-1)
        
        epsilon = 0.05 
        depth_diff = sampled_mesh_depths - bezier_depths + epsilon
        depth_sharpness = 5.0 
        depth_visibility = torch.sigmoid(depth_diff * depth_sharpness)
        
        # ---------------------------------------------------------
        # 4. COMBINE WITH LOCAL NORMALS (Optional)
        # ---------------------------------------------------------
        # Evaluate normals directly at the 3D Bezier control points
        current_normals = self.get_dynamic_normals(bezier_3d, mesh_verts, mesh_normals)
        view_dirs = torch.nn.functional.normalize(camera.eye.unsqueeze(0) - bezier_3d, p=2, dim=-1)
        dot_products = torch.sum(current_normals * view_dirs, dim=-1, keepdim=True).squeeze()
        
        normal_visibility = torch.sigmoid(dot_products * 10.0)
        
        # final_visibility = depth_visibility * normal_visibility
        final_visibility = depth_visibility 
        
        # ---------------------------------------------------------
        # 5. Final Width Calculation
        # ---------------------------------------------------------
        safe_depths = torch.clamp(bezier_depths, min=1e-3)
        perspective_scale = ref_dist / safe_depths
        
        k = 4.0
        final_widths = w_min + (w_max - w_min) * torch.sigmoid(k * (base_widths * perspective_scale - 0.5))
        
        # Apply strict occlusion mask to the Bezier widths
        final_widths = final_widths * final_visibility

        return bezier_2d, final_widths

    def get_serialized_scene(self, camera, view_w, view_h, mesh_verts, mesh_normals, mesh_depth, fill_color=None, stroke_color=None):
        """Converts 3D parameters to a 2D projected diffvg scene."""
        # Now returns the already converted 2D Bezier points
        bezier_2d, final_widths = self.forward(camera, view_w, view_h, mesh_verts, mesh_normals, mesh_depth)
        
        num_segments = self.num_kp - 1
        num_control_points = torch.full((num_segments,), 2, dtype=torch.int32, device=self.device)

        path = pydiffvg.Path(
            num_control_points=num_control_points,
            points=bezier_2d, 
            stroke_width=final_widths, 
            is_closed=False
        )

        f_color = fill_color if fill_color is not None else torch.tensor([0,0,0,0.0], device=self.device)
        s_color = stroke_color if stroke_color is not None else torch.tensor([0.1, 0.1, 0.1, 1.0], device=self.device)
        
        path_group = pydiffvg.ShapeGroup(
            shape_ids=torch.tensor([0], device=self.device),
            fill_color=f_color,
            stroke_color=s_color
        )
        
        return pydiffvg.RenderFunction.serialize_scene(
            view_w, view_h, [path], [path_group]
        )

    @torch.no_grad()
    def insert_keypoints(self, indices):
        """Stable 3D insertion."""
        new_pts = (self.points_3d[indices] + self.points_3d[indices + 1]) / 2.0
        new_widths = (self.unit_widths[indices] + self.unit_widths[indices + 1]) / 2.0
        
        updated_points = []
        updated_widths = []
        for i in range(self.num_kp):
            updated_points.append(self.points_3d[i:i+1])
            updated_widths.append(self.unit_widths[i:i+1])
            if i in indices:
                updated_points.append(new_pts[indices == i])
                updated_widths.append(new_widths[indices == i])
                
        self.num_kp = len(updated_points)
        self.points_3d = nn.Parameter(torch.cat(updated_points, dim=0))
        self.unit_widths = nn.Parameter(torch.cat(updated_widths, dim=0))
        self.register_buffer('G', self.get_gram_matrix_jerk(self.num_kp, self.device))

    def get_jerk_loss(self):
        """Smoothness penalty in 3D world space."""
        s_pos = torch.trace(self.points_3d.T @ self.G @ self.points_3d)
        s_width = torch.trace(self.unit_widths.T @ self.G @ self.unit_widths)
        return s_pos, s_width

    @staticmethod
    def get_gram_matrix_jerk(num_pts, device):
        D = torch.zeros((num_pts - 3, num_pts), device=device)
        for i in range(num_pts - 3):
            D[i, i:i+4] = torch.tensor([-1.0, 3.0, -3.0, 1.0], device=device)
        return D.T @ D

class UnitaryBSpline3D(nn.Module):
    def __init__(self, num_kp, kp_init, device):
        super().__init__()
        self.num_kp = num_kp
        self.device = device
        
        # Quintic Matrix (Keep as buffer)
        self.register_buffer('M_B', torch.tensor([
            [1/120, 26/120, 66/120, 26/120, 1/120, 0],
            [0, 10/120, 60/120, 40/120, 10/120, 0],
            [0, 0, 40/120, 60/120, 20/120, 0],
            [0, 0, 20/120, 60/120, 40/120, 0],
            [0, 0, 10/120, 40/120, 60/120, 10/120],
            [0, 1/120, 26/120, 66/120, 26/120, 1/120]
        ], dtype=torch.float32, device=device))

        # 1. PARAMETERS (Optimized in 3D Space)
        # kp_init should be [num_kp, 3]
        self.points_3d = nn.Parameter(kp_init.clone().detach().to(device))
        self.unit_widths = nn.Parameter(torch.full((num_kp, 1), 0.3, device=device))
        
        # Jerk Loss Matrix
        self.register_buffer('G', self.get_gram_matrix_jerk(num_kp, device))
    

    @torch.no_grad()
    def set_data(self, new_points, new_widths):
        """
        Safely updates the geometry and recalculates all dependent matrices (G).
        Use this instead of manually assigning self.points_3d.
        """
        device = self.device
        
        # 1. Sanity Checks
        # Ensure we are working with flat lists of points [N, 3]
        if new_points.dim() == 1: new_points = new_points.unsqueeze(0) # Handle single point case
        if new_widths.dim() == 1: new_widths = new_widths.unsqueeze(1)
        
        if new_points.shape[0] != new_widths.shape[0]:
            raise ValueError(f"Size Mismatch: {new_points.shape[0]} points vs {new_widths.shape[0]} widths")

        # 2. Update Scalar State
        self.num_kp = new_points.shape[0]

        # 3. Update Parameters (Wrap in nn.Parameter)
        # .contiguous() is crucial for memory layout after slicing/cat operations
        self.points_3d = nn.Parameter(new_points.contiguous().to(device))
        self.unit_widths = nn.Parameter(new_widths.contiguous().to(device))

        # 4. Rebuild the Jerk Matrix (G)
        # The matrix size is (N-3)xN, so it MUST be rebuilt when N changes.
        # We use register_buffer to ensure it moves with .to(device) and saves in state_dict
        new_G = self.get_gram_matrix_jerk(self.num_kp, device)
        self.register_buffer('G', new_G)


    def project_points(self, points_3d, mvp_matrix, width, height):
        """Projects 3D points to 2D screen space with Z-depth."""
        pts_homo = torch.cat([points_3d, torch.ones(points_3d.shape[0], 1, device=self.device)], dim=1)
        pts_clip = pts_homo @ mvp_matrix.T
        w = pts_clip[:, 3:4]
        pts_ndc = pts_clip[:, :2] / (w + 1e-6)
        
        screen_x = (pts_ndc[:, 0] + 1) * (width / 2)
        screen_y = (1 - pts_ndc[:, 1]) * (height / 2)
        return torch.stack([screen_x, screen_y], dim=1), w.squeeze()

    def to_bezier(self, keypoints):
        """Vectorized B-Spline to Bezier conversion."""
        # Padding for quintic (pass 2 points on each end)
        k = torch.cat([keypoints[0:1].repeat(2,1), keypoints, keypoints[-1:].repeat(2,1)], dim=0)
        segments = k.unfold(0, 6, 1) # [num_segments, dim, 6]
        bezier_segments = torch.matmul(segments, self.M_B.T)
        
        main_body = bezier_segments[:-1, :, :3]
        last_seg = bezier_segments[-1:, :, :]
        
        return torch.cat([
            main_body.transpose(1, 2).reshape(-1, keypoints.shape[-1]), 
            last_seg.transpose(1, 2).reshape(-1, keypoints.shape[-1])
        ], dim=0)
    
    def get_dynamic_normals(self, points_3d, mesh_verts, mesh_normals):
        """
        Finds the surface normal for each point by querying the nearest mesh vertex.
        Args:
            points_3d: [N, 3] tensor of B-spline control points
            mesh_verts: [V, 3] tensor of mesh vertices
            mesh_normals: [V, 3] tensor of mesh vertex normals
        Returns:
            dynamic_normals: [N, 3] tensor of normals at the current point locations
        """
        # 1. Compute pairwise distances between spline points and all mesh vertices
        # Shape: [N, V]
        distances = torch.cdist(points_3d, mesh_verts) 
        
        # 2. Find the index of the closest mesh vertex for each spline point
        # Shape: [N]
        nearest_indices = torch.argmin(distances, dim=1)
        
        # 3. Gather the normals of those closest vertices
        # Shape: [N, 3]
        dynamic_normals = mesh_normals[nearest_indices]
        
        return dynamic_normals

    def forward(self, camera, view_w, view_h, mesh_verts, mesh_normals, mesh_depth_map, ref_dist=5.0, w_min=0.2, w_max=6.5):
        mvp_matrix = camera.get_full_matrix()
        
        # 1. Project 3D Keypoints to 2D
        proj_2d, point_depths = self.project_points(
            self.points_3d, mvp_matrix, view_w, view_h
        )
        
        # ---------------------------------------------------------
        # 2. GLOBAL OCCLUSION CULLING (Depth Map Lookup)
        # ---------------------------------------------------------
        # Convert proj_2d (pixel coordinates) to normalized [-1, 1] coordinates for grid_sample
        # grid_sample expects (x, y) where -1 is top/left and 1 is bottom/right
        x_norm = (proj_2d[:, 0] / (view_w / 2.0)) - 1.0
        # Assuming your project_points flipped Y to match screen coords:
        y_norm = (proj_2d[:, 1] / (view_h / 2.0)) - 1.0 
        
        # Shape needed for grid_sample: [N, C, H, W] -> [1, 1, 1, num_points, 2]
        grid = torch.stack([x_norm, y_norm], dim=-1).view(1, 1, -1, 2)
        
        # Sample the depth map at the projected point coordinates
        # Using 'nearest' or 'bilinear' depending on your depth map resolution
        sampled_mesh_depths = F.grid_sample(
            mesh_depth_map, grid, mode='bilinear', padding_mode='border', align_corners=False
        ).view(-1) # Flatten back to [num_points]
        
        # Calculate Depth Visibility (Soft Step)
        # If point_depth < mesh_depth + epsilon, it is visible.
        epsilon = 0.05 # Tolerance to prevent z-fighting (curve intersecting surface)
        depth_diff = sampled_mesh_depths - point_depths + epsilon
        
        # Sigmoid creates a differentiable cliff. 
        # Points deeply occluded go to 0, visible points go to 1.
        depth_sharpness = 5.0 
        depth_visibility = torch.sigmoid(depth_diff * depth_sharpness)
        
        # ---------------------------------------------------------
        # 3. COMBINE WITH LOCAL NORMALS (Optional but recommended)
        # ---------------------------------------------------------
        # Normal culling is still great for smooth silhouette gradients!
        current_normals = self.get_dynamic_normals(self.points_3d, mesh_verts, mesh_normals)
        view_dirs = torch.nn.functional.normalize(camera.eye.unsqueeze(0) - self.points_3d, p=2, dim=-1)
        dot_products = torch.sum(current_normals * view_dirs, dim=-1, keepdim=True).squeeze()
        
        normal_visibility = torch.sigmoid(dot_products * 10.0)
        
        # Total visibility is the product of both masks
        final_visibility = depth_visibility #* normal_visibility
        
        # ---------------------------------------------------------
        # 4. Bezier Conversion & Final Widths
        # ---------------------------------------------------------
        bezier_pts = self.to_bezier(proj_2d)
        base_widths = self.to_bezier(self.unit_widths).squeeze()
        bezier_depths = self.to_bezier(point_depths.unsqueeze(-1)).squeeze()
        
        # Interpolate the combined visibility along the Bezier curve
        bezier_visibility = self.to_bezier(final_visibility.unsqueeze(-1)).squeeze()

        # Perspective Scaling
        safe_depths = torch.clamp(bezier_depths, min=1e-3)
        perspective_scale = ref_dist / safe_depths
        
        k = 4.0
        final_widths = w_min + (w_max - w_min) * torch.sigmoid(k * (base_widths * perspective_scale - 0.5))
        
        # Apply strict occlusion mask
        final_widths = final_widths * bezier_visibility

        return bezier_pts, final_widths

    def get_serialized_scene(self, camera, view_w, view_h, mesh_verts, mesh_normals, mesh_depth, fill_color=None, stroke_color=None):
        """
        Converts 3D Spline parameters to a 2D projected diffvg scene 
        based on the provided camera view.
        """
        # 1. Generate Projected Geometry (using the 3D forward pass)
        # This returns screen-space bezier_pts and depth-aware widths
        bezier_pts, final_widths = self.forward(camera, view_w, view_h, mesh_verts, mesh_normals, mesh_depth)
        
        # 2. Path Setup
        # Each B-spline segment (N-1) becomes a Bezier segment with 2 control points
        num_segments = self.num_kp - 1
        num_control_points = torch.full((num_segments,), 2, dtype=torch.int32, device=self.device)

        path = pydiffvg.Path(
            num_control_points=num_control_points,
            points=bezier_pts, 
            stroke_width=final_widths, 
            is_closed=False
        )

        # 3. Style Setup
        # Default to transparent fill and solid black stroke for high-contrast SDS updates
        f_color = fill_color if fill_color is not None else torch.tensor([0,0,0,0.0], device=self.device)
        s_color = stroke_color if stroke_color is not None else torch.tensor([0.1, 0.1, 0.1, 1.0], device=self.device)
        
        path_group = pydiffvg.ShapeGroup(
            shape_ids=torch.tensor([0], device=self.device), # Unique ID for this curve
            fill_color=f_color,
            stroke_color=s_color
        )
        
        # 4. Serialize with dynamic view dimensions
        return pydiffvg.RenderFunction.serialize_scene(
            view_w, view_h, [path], [path_group]
        )
    @torch.no_grad()
    def insert_keypoints(self, indices):
        """Stable 3D insertion."""
        new_pts = (self.points_3d[indices] + self.points_3d[indices + 1]) / 2.0
        new_widths = (self.unit_widths[indices] + self.unit_widths[indices + 1]) / 2.0
        
        # Build lists (Standard B-Spline midpoint logic)
        updated_points = []
        updated_widths = []
        for i in range(self.num_kp):
            updated_points.append(self.points_3d[i:i+1])
            updated_widths.append(self.unit_widths[i:i+1])
            if i in indices:
                updated_points.append(new_pts[indices == i])
                updated_widths.append(new_widths[indices == i])
                
        self.num_kp = len(updated_points)
        self.points_3d = nn.Parameter(torch.cat(updated_points, dim=0))
        self.unit_widths = nn.Parameter(torch.cat(updated_widths, dim=0))
        self.register_buffer('G', self.get_gram_matrix_jerk(self.num_kp, self.device))

    def get_jerk_loss(self):
        """Smoothness penalty in 3D world space."""
        # Penalizes high-frequency wiggles in X, Y, and Z
        s_pos = torch.trace(self.points_3d.T @ self.G @ self.points_3d)
        s_width = torch.trace(self.unit_widths.T @ self.G @ self.unit_widths)
        return s_pos, s_width

    @staticmethod
    def get_gram_matrix_jerk(num_pts, device):
        D = torch.zeros((num_pts - 3, num_pts), device=device)
        for i in range(num_pts - 3):
            D[i, i:i+4] = torch.tensor([-1.0, 3.0, -3.0, 1.0], device=device)
        return D.T @ D

In [2]:
import torch
import torch.nn.functional as F
import pydiffvg
import numpy as np
import torchvision.transforms as T
from PIL import Image
import os

# Ensure output directory exists
if not os.path.exists("fill_steps"):
    os.makedirs("fill_steps")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
canvas_width, canvas_height = 512, 512


def load_alpha_mask(image_path, size=512, device='cuda'):
    # 1. Load image and ensure it's in RGBA mode
    img = Image.open(image_path).convert('RGBA')
    
    # 2. Resize to match your canvas
    img = img.resize((size, size), resample=Image.LANCZOS)
    
    # 3. Convert to tensor [4, H, W]
    img_tensor = T.ToTensor()(img).to(device)
    
    # 4. Extract Alpha channel (index 3)
    # Alpha is usually 0.0 for transparent and 1.0 for opaque
    mask = img_tensor[3, :, :]
    
    return mask


# --- 2. Camera & Projection Functions ---
class DiffCamera:
    def __init__(self, eye, target, up, fov=45.0, aspect=1.0, near=0.1, far=100.0, device='cuda'):
        self.device = device
        self.eye = torch.tensor(eye, dtype=torch.float32, device=device)
        self.target = torch.tensor(target, dtype=torch.float32, device=device)
        self.up = torch.tensor(up, dtype=torch.float32, device=device)
        self.fov = fov
        self.aspect = aspect
        self.near = near
        self.far = far

    def get_view_matrix(self):
        z_axis = F.normalize(self.eye - self.target, dim=0)
        x_axis = F.normalize(torch.cross(self.up, z_axis), dim=0)
        y_axis = torch.cross(z_axis, x_axis)
        view_mat = torch.eye(4, device=self.device)
        view_mat[0, :3], view_mat[1, :3], view_mat[2, :3] = x_axis, y_axis, z_axis
        view_mat[0, 3] = -torch.dot(x_axis, self.eye)
        view_mat[1, 3] = -torch.dot(y_axis, self.eye)
        view_mat[2, 3] = -torch.dot(z_axis, self.eye)
        return view_mat

    def get_projection_matrix(self):
        tan_half_fov = torch.tan(torch.tensor(self.fov * 0.5 * (np.pi / 180.0)))
        proj_mat = torch.zeros((4, 4), device=self.device)
        proj_mat[0, 0] = 1 / (self.aspect * tan_half_fov)
        proj_mat[1, 1] = 1 / tan_half_fov
        proj_mat[2, 2] = -(self.far + self.near) / (self.far - self.near)
        proj_mat[2, 3] = -(2 * self.far * self.near) / (self.far - self.near)
        proj_mat[3, 2] = -1
        return proj_mat

    def get_full_matrix(self):
        return self.get_projection_matrix() @ self.get_view_matrix()

def project_points_mvp_with_depth(points_3d, mvp_matrix, width, height):
    pts_homo = torch.cat([points_3d, torch.ones(points_3d.shape[0], 1, device=device)], dim=1)
    pts_clip = pts_homo @ mvp_matrix.T
    w = pts_clip[:, 3:4]
    pts_ndc = pts_clip[:, :2] / (w + 1e-6)
    screen_x = (pts_ndc[:, 0] + 1) * (width / 2)
    screen_y = (1 - pts_ndc[:, 1]) * (height / 2)
    return torch.stack([screen_x, screen_y], dim=1), w.squeeze()


def initialize_3d_hull_tsp(masks, cameras, num_kp):
    # Simplified hull sampling: Grid search
    res = 40
    t = torch.linspace(-1.2, 1.2, res, device=device)
    grid = torch.stack(torch.meshgrid(t, t, t, indexing='ij'), dim=-1).reshape(-1, 3)
    
    inside_hull = torch.ones(grid.shape[0], dtype=torch.bool, device=device)
    for mask, cam in zip(masks, cameras):
        proj, _ = project_points_mvp_with_depth(grid, cam.get_full_matrix(), 512, 512)
        # Check if projected points fall within mask bounds
        px, py = proj[:, 0].long().clamp(0, 511), proj[:, 1].long().clamp(0, 511)
        inside_hull &= (mask[py, px] > 0.5)
    
    hull_pts = grid[inside_hull]
    if hull_pts.shape[0] < num_kp: # Fallback if hull is empty
        return torch.randn((num_kp, 3), device=device) * 0.5
    
    # Simple Greedy TSP for ordering
    pts_np = hull_pts[torch.randperm(hull_pts.shape[0])[:num_kp]].cpu().numpy()
    unvisited = list(range(len(pts_np)))
    curr = unvisited.pop(0)
    tour = [curr]
    while unvisited:
        dists = np.linalg.norm(pts_np[unvisited] - pts_np[curr], axis=1)
        curr = unvisited.pop(np.argmin(dists))
        tour.append(curr)
    return torch.tensor(pts_np[tour], device=device, dtype=torch.float32)

def get_jittered_coords(num_grid=4, patch_size=128, H=512, W=512):
    cell_h, cell_w = H // num_grid, W // num_grid
    coords = []
    for i in range(num_grid):
        for j in range(num_grid):
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            
            # Generate one random offset for this specific grid cell
            h_start = torch.randint(y0, max(y0 + 1, y1 - patch_size), (1,)).item()
            w_start = torch.randint(x0, max(x0 + 1, x1 - patch_size), (1,)).item()
            
            coords.append((h_start, w_start))
    return coords

def extract_patches_from_coords(img, coords, patch_size=128):
    if img.shape[-1] == 4: img = img[:, :, :3]
    img = img.permute(2, 0, 1).unsqueeze(0) # [1, 3, H, W]
    
    patches = []
    for h_start, w_start in coords:
        patch = img[:, :, h_start:h_start+patch_size, w_start:w_start+patch_size]
        patch = F.interpolate(patch, (224, 224), mode='bicubic', align_corners=False)
        patches.append(patch)
    return torch.cat(patches, dim=0)

@torch.no_grad()
def redistribute_capacity(model, optimizer_point, optimizer_width, 
                          width_threshold=0.1, 
                          min_consecutive_len=10, 
                          v_reset_factor=0.1):
    
    device = model.points_3d.device
    
    # 1. Safety Check
    if model.points_3d.grad is None:
        return

    # Handle NaNs in gradients
    grads = model.points_3d.grad
    grads = torch.nan_to_num(grads, nan=0.0)
    old_grads = grads.norm(dim=1)

    # ==========================================
    # PHASE 1: PRUNING (Run-Length Logic)
    # ==========================================
    
    widths = model.unit_widths.data.view(-1) 
    is_dead = widths < width_threshold
    
    change_points = is_dead[1:] != is_dead[:-1]
    block_ids = torch.cat([torch.tensor([0], device=device), change_points.long()]).cumsum(dim=0)
    block_sizes = torch.bincount(block_ids)
    point_block_sizes = block_sizes[block_ids]
    
    to_prune = is_dead & (point_block_sizes >= min_consecutive_len)
    
    to_prune[0] = False
    to_prune[-1] = False
    
    survivor_mask = ~to_prune
    num_freed_points = to_prune.sum().item()
    
    if survivor_mask.sum() < 4:
        print("Skipping prune: Too few survivors.")
        return

    if num_freed_points > 0:
        print(f"Pruning {num_freed_points} points...")

    # Slice Data
    surviving_points = model.points_3d.data[survivor_mask]
    surviving_widths = model.unit_widths.data[survivor_mask]
    surviving_grads = old_grads[survivor_mask]

    # Slice Optimizer States
    old_p_state = optimizer_point.state.get(model.points_3d, {})
    old_w_state = optimizer_width.state.get(model.unit_widths, {})

    def slice_state(state, mask):
        if not state: return {}
        return {
            'step': state['step'], 
            'exp_avg': state['exp_avg'][mask],
            'exp_avg_sq': state['exp_avg_sq'][mask]
        }

    new_p_state_dict = slice_state(old_p_state, survivor_mask)
    new_w_state_dict = slice_state(old_w_state, survivor_mask)

    # ==========================================
    # PHASE 2: RE-INVESTMENT
    # ==========================================
    
    new_seg_vecs = surviving_points[1:] - surviving_points[:-1]
    new_seg_lens = torch.norm(new_seg_vecs, dim=1)
    
    segment_stress = torch.max(surviving_grads[:-1], surviving_grads[1:])
    segment_stress[new_seg_lens < 1e-4] = -1.0
    
    num_candidates = (segment_stress > 0).sum().item()
    k_budget = min(num_freed_points, num_candidates)
    
    if k_budget > 0:
        _, insertion_indices = torch.topk(segment_stress, k_budget)
        insertion_indices = insertion_indices.sort()[0]
    else:
        insertion_indices = torch.tensor([], device=device, dtype=torch.long)

    # Insertion with Jitter
    def insert_with_jitter(param_data, state_dict, indices, is_width=False):
        if len(indices) == 0: return param_data, state_dict
        
        N = param_data.shape[0]
        new_data_list = []
        
        has_state = len(state_dict) > 0
        new_m_list = []
        new_v_list = []
        
        if has_state:
            old_m = state_dict['exp_avg']
            old_v = state_dict['exp_avg_sq']

        for i in range(N):
            new_data_list.append(param_data[i:i+1])
            if has_state:
                new_m_list.append(old_m[i:i+1])
                new_v_list.append(old_v[i:i+1])
            
            if i in indices:
                next_i = min(i + 1, N - 1)
                
                mid_val = (param_data[i] + param_data[next_i]) / 2.0
                
                if not is_width:
                    noise = torch.randn_like(mid_val) * 1e-4
                    mid_val = mid_val + noise
                
                new_data_list.append(mid_val.unsqueeze(0))
                
                if has_state:
                    # CRITICAL FIX: Reset momentum for new points
                    mid_m = torch.zeros_like(old_m[i]) 
                    mid_v = (old_v[i] + old_v[next_i]) / 2.0 * v_reset_factor
                    
                    new_m_list.append(mid_m.unsqueeze(0))
                    new_v_list.append(mid_v.unsqueeze(0))
        
        final_data = torch.cat(new_data_list)
        final_state = {}
        if has_state:
            final_state = {
                'step': state_dict['step'], 
                'exp_avg': torch.cat(new_m_list),
                'exp_avg_sq': torch.cat(new_v_list)
            }
        return final_data, final_state

    final_p_data, final_p_state = insert_with_jitter(surviving_points, new_p_state_dict, insertion_indices, is_width=False)
    final_w_data, final_w_state = insert_with_jitter(surviving_widths, new_w_state_dict, insertion_indices, is_width=True)

    # ==========================================
    # PHASE 3: FINAL COMMIT (MODIFIED)
    # ==========================================
    
    if torch.isnan(final_p_data).any():
        final_p_data = torch.nan_to_num(final_p_data, nan=0.5)
    
    final_p_data += torch.randn_like(final_p_data) * 1e-6

    # 1. Capture Old Parameters (for optimizer cleanup)
    old_p_param = model.points_3d
    old_w_param = model.unit_widths
    
    # 2. UPDATE MODEL (Using the safe setter!)
    # This automatically updates num_kp and recalculates Matrix G
    model.set_data(final_p_data, final_w_data)
    
    # 3. Update Optimizers
    # Remove old params from internal state
    if old_p_param in optimizer_point.state: del optimizer_point.state[old_p_param]
    if old_w_param in optimizer_width.state: del optimizer_width.state[old_w_param]
    
    # Inject new state (if it exists)
    if final_p_state: optimizer_point.state[model.points_3d] = final_p_state
    if final_w_state: optimizer_width.state[model.unit_widths] = final_w_state
    
    # Point optimizer to the NEW parameters created by set_data
    optimizer_point.param_groups[0]['params'] = [model.points_3d]
    optimizer_width.param_groups[0]['params'] = [model.unit_widths]
    
    print(f"Redistribution Complete: {len(old_p_param)} -> {model.num_kp} points")

def get_gradient_driven_indices(model, k_budget, min_segment_length=0.01):
    """
    Selects the top 'k_budget' segments with the highest gradient magnitude.
    Includes a safety check to avoid splitting segments that are already too tiny.
    """
    if k_budget <= 0 or model.points_3d.grad is None:
        return torch.tensor([], dtype=torch.long, device=model.device)

    # 1. Calculate Gradient Magnitudes per Point
    # [N, 3] -> [N]
    grads = model.points_3d.grad.norm(dim=1)
    
    # 2. Map Point Gradients to Segments
    # A segment's "stress" is defined by the max gradient of its two endpoints
    # [N-1] segments
    segment_stress = torch.max(grads[:-1], grads[1:])
    
    # 3. SAFETY: Penalize tiny segments
    # We do NOT want to add points to a segment that is already microscopic.
    # It causes numerical explosions.
    points = model.points_3d.detach()
    segment_lengths = torch.norm(points[1:] - points[:-1], dim=1)
    
    # Zero out the stress score for tiny segments so they aren't picked
    # (Setting to -1 ensures they are at the bottom of the top-k list)
    segment_stress[segment_lengths < min_segment_length] = -1.0
    
    # 4. Select Top K
    # If we have fewer valid segments than k_budget, take what we can
    num_valid = (segment_stress > 0).sum()
    k = min(k_budget, num_valid)
    
    if k == 0:
        return torch.tensor([], dtype=torch.long, device=model.device)

    # indices are the segment indices (0 to N-2)
    _, indices = torch.topk(segment_stress, k)
    
    # Sort indices because insertion usually requires sorted order
    return indices.sort()[0]

# Process OBJ

In [3]:
import numpy as np
import networkx as nx
import torch
from rdp import rdp

def load_ordered_sfc(obj_path, simplify=True, epsilon=0.01):
    vertices = []
    edges = []
    
    with open(obj_path, 'r') as f:
        for line in f:
            parts = line.split()
            if not parts: continue
            if parts[0] == 'v':
                vertices.append([float(x) for x in parts[1:4]])
            elif parts[0] == 'l':
                # OBJ indices are 1-based
                v_indices = [int(i) - 1 for i in parts[1:]]
                for i in range(len(v_indices) - 1):
                    edges.append((v_indices[i], v_indices[i+1]))

    # Create a graph to find the continuous path
    G = nx.Graph()
    G.add_edges_from(edges)
    
    # SFCs are usually a single connected component
    # We find the path by traversing from one end (or any node if it's a loop)
    nodes = list(nx.dfs_preorder_nodes(G, source=edges[0][0]))
    
    ordered_pts = np.array([vertices[i] for i in nodes])
    corrected_pts = ordered_pts[:, [2, 0, 1]] # Swap Y and Z for our coordinate system
    if simplify:
        corrected_pts = rdp(corrected_pts, epsilon=epsilon)
    return corrected_pts



In [156]:
raw_points = load_ordered_sfc('SF/curves/bob/003.obj', simplify=True, epsilon=0.01)

In [5]:
from scipy.interpolate import splprep

def fit_bspline_to_points(simplified_pts, target_num_ctrl_pts=1500, degree=3, is_closed=True, device="cuda"):
    """
    Converts a sequence of 3D points into B-spline control points.
    
    Args:
        simplified_pts (np.ndarray): (N, 3) array of ordered vertices.
        target_num_ctrl_pts (int): The number of control points for the optimizer.
        degree (int): B-spline degree (3 = cubic).
        is_closed (bool): Whether the SFC is a closed loop.
        
    Returns:
        torch.Tensor: (target_num_ctrl_pts, 3) control points with gradients enabled.
    """
    
    # Transpose for splprep: expects [x_arr, y_arr, z_arr]
    data = [simplified_pts[:, 0], simplified_pts[:, 1], simplified_pts[:, 2]]
    
    # splprep finds the B-spline representation
    # s=0 forces the spline to pass exactly through all simplified_pts
    # per=is_closed ensures the periodic boundary condition
    tck, u = splprep(data, s=0, k=degree, per=is_closed)
    
    # tck[1] contains the control points (the coefficients 'c')
    # These are stored as [array_x, array_y, array_z]
    ctrl_pts_x = np.array(tck[1][0])
    ctrl_pts_y = np.array(tck[1][1])
    ctrl_pts_z = np.array(tck[1][2])
    
    # Stack into (N, 3)
    ctrl_pts_np = np.stack([ctrl_pts_x, ctrl_pts_y, ctrl_pts_z], axis=1)
    
    # --- Resampling Control Points ---
    # splprep might return a number of control points that doesn't match 
    # your target. We resample the control polygon to your exact target count.
    if len(ctrl_pts_np) != target_num_ctrl_pts:
        from scipy.interpolate import interp1d
        lin_x = np.linspace(0, 1, len(ctrl_pts_np))
        new_x = np.linspace(0, 1, target_num_ctrl_pts)
        f = interp1d(lin_x, ctrl_pts_np, axis=0, kind='linear')
        ctrl_pts_np = f(new_x)

    # Convert to PyTorch Tensor
    control_points = torch.tensor(ctrl_pts_np, dtype=torch.float32)
    
    return control_points.clone().detach().to(device).requires_grad_(True)

In [6]:
import open_clip

model, preprocess_train, preprocess_test = open_clip.create_model_and_transforms(
    'ViT-B/32',
    pretrained='CLIPAG_ViTB32.pt',
    precision="amp",
    device="cuda",
    weights_only=False
)

tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [7]:
import torch.nn.functional as F

import math
import imageio
def save_gif(frames, path, fps=10):
    processed_frames = []
    
    for frame in frames:
        if torch.is_tensor(frame):
            img = frame.detach().cpu().numpy()
        else:
            img = frame.copy()

        if img.shape[0] <= 4 and img.shape[0] < img.shape[1]:
            img = np.transpose(img, (1, 2, 0))

        # 1. ALPHA BLENDING (Stay in Linear Space for the math)
        if img.shape[-1] == 4:
            alpha = img[:, :, 3:4]
            rgb = img[:, :, :3] # Keep this linear (0.1 stays 0.1)
            bg_color = np.array([1.0, 1.0, 1.0], dtype=img.dtype)
            img = rgb * alpha + bg_color * (1 - alpha)
        
        # 2. GAMMA CORRECTION (Linear -> sRGB for display)
        # We apply this to the WHOLE image after blending
        # This turns 0.1 into ~0.35, preserving your grayish look
        img = np.power(img, 1/2.2) 
        
        # 3. Scale to 0-255 uint8
        img = np.clip(img, 0, 1)
        img = (img * 255).astype(np.uint8)
        processed_frames.append(img)

    imageio.mimsave(path, processed_frames, fps=fps, loop=0)

def render_360_view(bspline, num_frames=45, radius=3.0):
    """
    Rotates the camera around the Y-axis and renders the 3D curve
    with perspective-aware stroke widths.
    """
    if not os.path.exists("render_360"):
        os.makedirs("render_360")

    frames = []
    with torch.no_grad():
        for i in range(num_frames):
            t = i / num_frames
            
            # 1. Horizontal rotation (Around Z-axis)
            # 2 * pi for a full circle
            azimuth = 2 * math.pi * t
            
            # 2. Vertical oscillation (Tilt / Around X-axis)
            # Use a sine wave to go from eye-level -> top view -> eye-level
            # We add a small offset (0.2) so we don't look from perfectly underneath
            elevation = math.pi/4 * math.sin(2 * math.pi * t) + 0.5
            
            # Spherical to Cartesian conversion
            eye_x = radius * math.cos(elevation) * math.cos(azimuth)
            eye_y = radius * math.cos(elevation) * math.sin(azimuth)
            eye_z = radius * math.sin(elevation)
            
            temp_cam = DiffCamera(eye=[eye_x, eye_y, eye_z], 
                                  target=[0.0, 0.0, 0.0], 
                                  up=[0.0, 0.0, 1.0],
                                  fov=55.0, 
                                  device=device)
            mesh_depth = render_native_depth_map(mesh_vertices, temp_cam, 512, 512)
            scene_args = bspline.get_serialized_scene(temp_cam, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth)
            img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
            frames.append(img.cpu())
            # Save frame
            pydiffvg.imwrite(img.cpu(), f"fill_steps/ frames/frame_{i:03d}.png")
            print(f"Rendering frame {i+1}/{num_frames}", end='\r')
    save_gif(frames, "fill_steps/optimization_360.gif", fps=10)

def get_patches(img, num_patches=16, patch_size=128):
    # img: [H, W, 4] -> [1, 3, H, W]
    if img.shape[-1] == 4:
        img = img[:, :, :3]
    img = img.permute(2, 0, 1).unsqueeze(0)
    
    H, W = img.shape[2], img.shape[3]
    patches = []
    
    # Use a fixed set of random coordinates for this specific step
    for _ in range(num_patches):
        h_start = torch.randint(0, H - patch_size, (1,))
        w_start = torch.randint(0, W - patch_size, (1,))
        
        patch = img[:, :, h_start:h_start+patch_size, w_start:w_start+patch_size]
        # Resize to CLIP's 224x224
        patch = F.interpolate(patch, (224, 224), mode='bicubic', align_corners=False)
        patches.append(patch)
        
    return torch.cat(patches, dim=0) # [num_patches, 3, 224, 224]

def calculate_directional_loss_patch(current_patches, source_patches, text_direction, model):
    # Standard CLIP Normalization constants
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).to(current_patches.device).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).to(current_patches.device).view(1, 3, 1, 1)
    
    curr = (current_patches - mean) / std
    src = (source_patches - mean) / std
    
    # 1. Encode all patches
    # current_embeds shape: [n_patches, 512]
    current_embeds = model.encode_image(curr)
    source_embeds = model.encode_image(src)
    
    # 2. Normalize individual embeddings
    current_embeds = current_embeds / (current_embeds.norm(dim=-1, keepdim=True) + 1e-8)
    source_embeds = source_embeds / (source_embeds.norm(dim=-1, keepdim=True) + 1e-8)
    
    # 3. Calculate Directions for each patch
    image_dirs = current_embeds - source_embeds
    mag = image_dirs.norm(dim=-1, keepdim=True)
    image_dirs = image_dirs / (mag + 1e-8)
    
    # 4. Cosine similarity against the text_direction
    # text_direction: [512] -> [1, 512] for broadcasting
    similarities = torch.cosine_similarity(image_dirs, text_direction.unsqueeze(0), dim=-1)
    
    # Mask out patches where no change happened (prevents NaN/noise)
    valid_mask = (mag > 1e-7).float()
    if valid_mask.sum() == 0:
        return torch.tensor(0.0, device=current_patches.device, requires_grad=True)
        
    return 1 - (similarities * valid_mask.squeeze()).sum() / (valid_mask.sum() + 1e-8)

def get_text_direction(model, tokenizer, target_prompt, source_prompt, device):
    """
    Computes the normalized semantic direction vector between two prompts.
    """
    with torch.no_grad():
        # 1. Tokenize both prompts
        tokens = tokenizer([source_prompt, target_prompt]).to(device)
        
        # 2. Encode to CLIP latent space
        # model.encode_text returns [2, 512] or [2, 768] depending on model size
        text_features = model.encode_text(tokens)
        
        # 3. Normalize individual embeddings
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # 4. Extract Source and Target
        source_embed = text_features[0]
        target_embed = text_features[1]
        
        # 5. Compute the Direction Vector (Target - Source)
        direction = target_embed - source_embed
        
        # 6. Normalize the Direction Vector
        # This ensures the loss magnitude is consistent regardless of prompt length
        direction /= direction.norm(dim=-1, keepdim=True)
        
    return direction



# Helper to prepare diffvg output for CLIP
def preprocess_for_clip(img, target_size=224):
    # 1. Handle RGBA -> RGB if necessary
    if img.shape[-1] == 4:
        img = img[:, :, :3]
    
    # 2. Permute [H, W, C] -> [1, C, H, W]
    img = img.permute(2, 0, 1).unsqueeze(0)
    
    # 3. Resize to CLIP's native 224x224
    img = F.interpolate(img, size=(target_size, target_size), mode='bicubic', align_corners=False)
    
    # 4. Standard CLIP Normalization
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).to(img.device).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).to(img.device).view(1, 3, 1, 1)
    return (img - mean) / std


In [157]:
import trimesh

def load_and_densify_mesh(path, max_edge=0.1):
    # 1. Load the original mesh
    mesh = trimesh.load(path)
    
    # 2. Subdivide until the mesh is dense enough
    v, f = trimesh.remesh.subdivide_to_size(mesh.vertices, mesh.faces, max_edge=max_edge)
    
    # 3. Reconstruct a dense Trimesh object
    # Trimesh automatically computes smooth vertex_normals upon creation based on the faces
    dense_mesh = trimesh.Trimesh(vertices=v, faces=f)
    
    # 4. Extract the raw numpy arrays
    raw_vertices = dense_mesh.vertices
    raw_normals = dense_mesh.vertex_normals
    
    # 5. Apply the axis permutation to BOTH vertices and normals
    v_corrected = raw_vertices[:, [2, 0, 1]]
    n_corrected = raw_normals[:, [2, 0, 1]] 
    
    # 6. Convert to PyTorch tensors and move to GPU
    v_tensor = torch.tensor(v_corrected, dtype=torch.float32).cuda()
    n_tensor = torch.tensor(n_corrected, dtype=torch.float32).cuda()
    
    # 7. Normalize the normal vectors just to be mathematically safe
    n_tensor = torch.nn.functional.normalize(n_tensor, p=2, dim=-1)
    
    return v_tensor, n_tensor

mesh_vertices, normal = load_and_densify_mesh("SF/curves/bob/mesh.obj", max_edge=0.03)

def simple_mesh_fidelity_loss(control_points, mesh_vertices):
    """|
    Computes loss as the distance from each control point to its 
    closest neighbor in the fixed mesh vertex set.
    """
    # control_points: (N, 3)
    # mesh_vertices: (M, 3)
    
    # Compute squared L2 distance matrix (N, M)
    # Using the expansion: (a-b)^2 = a^2 - 2ab + b^2
    dist_sq = torch.cdist(control_points.unsqueeze(0), mesh_vertices.unsqueeze(0), p=2).squeeze(0)**2
    
    # For each control point, find the distance to the closest mesh vertex
    min_dist, _ = torch.min(dist_sq, dim=1)
    
    
    return torch.mean(min_dist)

def render_native_depth_map(mesh_verts, camera, view_w, view_h):
    """
    Creates a depth map by projecting dense mesh vertices into a 2D Z-buffer.
    Uses zero external dependencies.
    """
    device = mesh_verts.device
    mvp_matrix = camera.get_full_matrix()
    
    # 1. Project all mesh vertices to 2D
    pts_homo = torch.cat([mesh_verts, torch.ones(mesh_verts.shape[0], 1, device=device)], dim=1)
    pts_clip = pts_homo @ mvp_matrix.T
    w = pts_clip[:, 3:4]
    pts_ndc = pts_clip[:, :2] / (w + 1e-6)
    
    # Convert NDC to Pixel Coordinates
    screen_x = ((pts_ndc[:, 0] + 1.0) * (view_w / 2.0)).long()
    screen_y = ((1.0 - pts_ndc[:, 1]) * (view_h / 2.0)).long()
    depths = w.squeeze()
    
    # 2. Filter out points that are behind the camera or off-screen
    valid_mask = (screen_x >= 0) & (screen_x < view_w) & \
                 (screen_y >= 0) & (screen_y < view_h) & \
                 (depths > 0)
                 
    sx = screen_x[valid_mask]
    sy = screen_y[valid_mask]
    s_depths = depths[valid_mask]
    
    # 3. The "Painter's Algorithm" (Z-Buffering)
    # We sort the points from FURTHEST to CLOSEST. 
    # When we assign them to the image, the closest points overwrite the furthest ones.
    sorted_indices = torch.argsort(s_depths, descending=True)
    sx = sx[sorted_indices]
    sy = sy[sorted_indices]
    s_depths = s_depths[sorted_indices]
    
    # 4. Create the blank canvas (initialized to infinity)
    depth_buffer = torch.full((view_h * view_w,), 10000.0, device=device)
    
    # 5. Splat the points onto the canvas
    # Convert 2D coordinates to 1D flat indices
    flat_indices = sy * view_w + sx
    depth_buffer[flat_indices] = s_depths
    
    # Reshape back to image format [1, 1, H, W]
    depth_map = depth_buffer.view(1, 1, view_h, view_w)
    
    # 6. Fill Micro-Holes (Dilation via Min-Pooling)
    # Because we are projecting points instead of solid triangles, there might be tiny 1-pixel gaps.
    # We apply a 3x3 min-filter. In PyTorch, min-pool is just negative max-pool.
    splat_size = 5
    padding = splat_size // 2
    
    # We pool the negative depth map to smear the CLOSEST depths over the gaps
    depth_map = -F.max_pool2d(-depth_map, kernel_size=splat_size, stride=1, padding=padding)
    
    return depth_map

def render_silhouette_mask(mesh_verts, camera, view_w, view_h, splat_size=21):
    """
    Renders a solid 2D area contour (silhouette) of a 3D point cloud.
    Uses Morphological Closing to seal holes caused by low point density.
    """
    device = mesh_verts.device
    mvp_matrix = camera.get_full_matrix()
    
    # 1. Project points to 2D
    pts_homo = torch.cat([mesh_verts, torch.ones(mesh_verts.shape[0], 1, device=device)], dim=1)
    pts_clip = pts_homo @ mvp_matrix.T
    w = pts_clip[:, 3:4]
    pts_ndc = pts_clip[:, :2] / (w + 1e-6)
    
    # Convert NDC to Screen Coordinates
    screen_x = ((pts_ndc[:, 0] + 1.0) * (view_w / 2.0)).long()
    screen_y = ((1.0 - pts_ndc[:, 1]) * (view_h / 2.0)).long()
    
    # Filter valid points (on-screen and in front of camera)
    valid_mask = (screen_x >= 0) & (screen_x < view_w) & \
                 (screen_y >= 0) & (screen_y < view_h) & \
                 (w.squeeze() > 0)
                 
    sx = screen_x[valid_mask]
    sy = screen_y[valid_mask]
    
    # 2. Create the blank canvas (0.0 for background)
    mask_buffer = torch.zeros((view_h * view_w,), dtype=torch.float32, device=device)
    
    # 3. Splat points (1.0 for foreground)
    flat_indices = sy * view_w + sx
    mask_buffer[flat_indices] = 1.0
    
    # Reshape to [Batch, Channel, Height, Width] for pooling
    mask_map = mask_buffer.view(1, 1, view_h, view_w)
    
    # ---------------------------------------------------------
    # 4. Morphological Closing (Fill holes without bloating)
    # ---------------------------------------------------------
    padding = splat_size // 2
    
    # STEP A: Dilation (Max Pooling) - Closes the interior holes
    dilated = F.max_pool2d(mask_map, kernel_size=splat_size, stride=1, padding=padding)
    
    # STEP B: Erosion (Min Pooling) - Shrinks the outer edge back to normal
    # In PyTorch, Erosion is just negative Max Pooling on negative data
    closed_mask = -F.max_pool2d(-dilated, kernel_size=splat_size, stride=1, padding=padding)
    
    return closed_mask


def save_depth_map(depth_tensor, filename="depth_map.png"):
    """
    Normalizes a depth map tensor (containing inf for background) 
    and saves it as a grayscale image.
    
    Args:
        depth_tensor: [1, 1, H, W] tensor from our native renderer
        filename: String path to save the image
    """
    # 1. Squeeze out the batch and channel dimensions to get [H, W]
    # Clone it so we don't accidentally modify the tensor used for optimization
    d = depth_tensor.squeeze().clone().detach()
    
    # 2. Identify the valid foreground pixels (ignore infinity)
    valid_mask = d != float('inf')
    
    if not valid_mask.any():
        print(f"Warning: Depth map is completely empty. Is the camera pointing at the mesh?")
        return

    # 3. Find the minimum (closest) and maximum (furthest) depths of the object
    min_depth = d[valid_mask].min()
    max_depth = d[valid_mask].max()
    
    # 4. Normalize the foreground to a 0.0 to 1.0 range
    # Adding a tiny epsilon (1e-6) prevents division by zero if the object is perfectly flat
    d[valid_mask] = (d[valid_mask] - min_depth) / (max_depth - min_depth + 1e-6)
    
    # Optional but highly recommended: Invert the depth for visualization
    # This makes the closest parts of the mesh WHITE and the furthest parts DARK GRAY.
    d[valid_mask] = 1.0 - d[valid_mask]
    
    # 5. Set the infinite background pixels to pure BLACK (0.0)
    d[~valid_mask] = 0.0
    
    # 6. Convert to an 8-bit integer numpy array [0, 255]
    d_numpy = (d.cpu().numpy() * 255).astype(np.uint8)
    
    # 7. Save using PIL
    img = Image.fromarray(d_numpy, mode='L') # 'L' mode is for grayscale
    img.save(filename)
    print(f"Saved depth map to: {filename}")
#save_depth_map(mesh_depth_map, filename="mesh_depth_map.png")

def save_silhouette_mask(mask_tensor, filename="silhouette_mask.png"):
    """
    Converts the binary [0.0, 1.0] mask tensor into an image and saves it.
    
    Args:
        mask_tensor: [1, 1, H, W] tensor from render_silhouette_mask
        filename: String path to save the image
    """
    # 1. Squeeze out the batch and channel dimensions to get [H, W]
    # Clone and detach so we don't break any computation graphs
    m = mask_tensor.squeeze().clone().detach().cpu()
    
    # 2. Convert to an 8-bit integer numpy array
    # Since our mask is strictly 0.0 or 1.0, multiplying by 255 gives us pure black and white
    m_numpy = (m.numpy() * 255).clip(0, 255).astype(np.uint8)
    
    # 3. Save using PIL in Grayscale ('L') mode
    img = Image.fromarray(m_numpy, mode='L')
    img.save(filename)
    print(f"Saved solid silhouette mask to: {filename}")

In [158]:
print("Total raw points loaded:", raw_points.shape[0])
processed_points_number = raw_points.shape[0]
target_points = 1100
keypoints = fit_bspline_to_points(raw_points, target_num_ctrl_pts=target_points)
#keypoint_widths = torch.full((processed_points_number, 1), 2.0, device=device, requires_grad=True)

Total raw points loaded: 3860


In [209]:
canvas_width = 512
canvas_height = 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'

bspline = UnitaryBSpline3D_New(num_kp=target_points, 
                       kp_init=keypoints, 
                       device='cuda')

opt_bspline = torch.optim.Adam([bspline.points_3d], lr=0.004)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt_bspline, T_max=iter)

# Group 2: Other parameters (to stay constant)
opt_fixed = torch.optim.Adam([bspline.unit_widths], lr=0.3)

text_dir = get_text_direction(model, tokenizer, "infinity symbol", "A continuous 3D wire drawing", "cuda")

In [210]:
critical_cam1 = DiffCamera(eye=[-3.0, 0.0, 0.3], target=[0.0, 0.0, 0.0], up=[0.0, 0.0, 1.0], fov=55.0, device=device)
critical_cam2 = DiffCamera(eye=[0.0, -3.0, 0.3], target=[0.0, 0.0, 0.0], up=[0.0, 0.0, 1.0], fov=55.0, device=device)
critical_cam3 = DiffCamera(eye=[-0.1, -0.0, 3.0], target=[0.0, 0.0, 0.0], up=[0.0, 0.0, 1.0], fov=55.0, device=device)
critical_cam4 = DiffCamera(eye=[0.0, 3.0, 0.3], target=[0.0, 0.0, 0.0], up=[0.0, 0.0, 1.0], fov=55.0, device=device)
critical_cam5 = DiffCamera(eye=[3.0, 0.0, 0.3], target=[0.0, 0.0, 0.0], up=[0.0, 0.0, 1.0], fov=55.0, device=device)
critical_cams = [critical_cam1, critical_cam2, critical_cam3, critical_cam4, critical_cam5]
with torch.no_grad():
    mesh_depth = render_native_depth_map(mesh_vertices, critical_cam2, 512, 512)
    scene_args = bspline.get_serialized_scene(critical_cam2, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth=mesh_depth)
    source_img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args).detach()
    pydiffvg.imwrite(source_img.cpu(), f"fill_steps/init.png")

In [211]:
def multi_resolution_mse_loss(rendered_img, target_img, levels=4):
    """
    Computes MSE across multiple resolution scales.
    Args:
        rendered_img: [1, C, H, W] tensor from pydiffvg
        target_img: [1, C, H, W] tensor of the target mesh render
        levels: Number of downsampling steps
    """
    total_loss = 0.0
    weight = 1.0
    
    curr_render = rendered_img
    curr_target = target_img
    
    for i in range(levels):
        # 1. Calculate MSE at the current resolution
        total_loss += weight * F.mse_loss(curr_render, curr_target)
        
        # 2. Downsample by a factor of 2 for the next level
        if i < levels - 1:
            curr_render = F.avg_pool2d(curr_render, kernel_size=2, stride=2)
            curr_target = F.avg_pool2d(curr_target, kernel_size=2, stride=2)
            
            # Optional: Increase weight of lower resolutions to force global coverage first
            weight *= 1.5 
            
    return total_loss

def get_perturbed_critical_view(device, perturbation_deg=15.0):
    """
    Selects one of 3 critical views and perturbs it slightly.
    Maintains the distance to the target (radius).
    """
    # 1. Define the 3 Critical Centers (Eyes)
    critical_eyes = [
        [-3.0, 0.0, 0.3],   # Cam 1
        [0.0, -3.0, 0.3],   # Cam 2
        [0.1, 0.0, 3.0],    # Cam 3
        [0.0, 3.0, 0.3],    # Cam 3
        [3.0, 0.0, 0.3]    # Cam 3
    ]
    
    # 2. Randomly select one index
    idx = np.random.randint(0, len(critical_eyes))
    center_eye = np.array(critical_eyes[idx])
    
    # 3. Convert Cartesian (x,y,z) -> Spherical (r, theta, phi)
    # Assuming Z is UP based on your 'up=[0,0,1]'
    x, y, z = center_eye
    r = np.linalg.norm(center_eye)
    
    # Theta (Azimuth): Angle in XY plane (0 to 2pi)
    theta = np.arctan2(y, x)
    
    # Phi (Inclination): Angle from Z-axis (0 is Top, pi is Bottom)
    # z = r * cos(phi) => phi = arccos(z/r)
    phi = np.arccos(z / r)
    
    # 4. Apply Random Perturbation (in Radians)
    pert_rad = np.deg2rad(perturbation_deg)
    
    # Random offset between [-pert, +pert]
    d_theta = np.random.uniform(-pert_rad, pert_rad)
    d_phi = np.random.uniform(-pert_rad, pert_rad)
    
    new_theta = theta + d_theta
    new_phi = np.clip(phi + d_phi, 0.01, np.pi - 0.01) # Avoid gimbal lock at poles
    
    # 5. Convert Spherical -> Cartesian
    # Standard Physics Convention (Z-up)
    new_x = r * np.sin(new_phi) * np.cos(new_theta)
    new_y = r * np.sin(new_phi) * np.sin(new_theta)
    new_z = r * np.cos(new_phi)
    
    new_eye = [new_x, new_y, new_z]

    # 6. Create Camera
    cam = DiffCamera(
        eye=new_eye,
        target=[0.0, 0.0, 0.0],
        up=[0.0, 0.0, 1.0],
        fov=45.0,
        device=device
    )
    return cam

In [212]:
# --- Loop ---
for step in range(251):
    # if step == 100:
    #     print("\n>>> Triggering Adaptive Resampling: Doubling Resolution")
    #     redistribute_capacity(bspline, opt_bspline, opt_fixed)
    opt_bspline.zero_grad()
    opt_fixed.zero_grad()
    
    cam = get_perturbed_critical_view(device, perturbation_deg=20.0)
    # 1. Get the "Intrinsic" widths (the optimized thickness of the wire)
    # This remains consistent across all views
    mesh_depth = render_native_depth_map(mesh_vertices, cam, 512, 512)
    objmask = render_silhouette_mask(mesh_vertices, cam, view_w=512, view_h=512, splat_size=7)
    scene_args = bspline.get_serialized_scene(cam, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth)
    img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
    
    coords = get_jittered_coords(num_grid=4, patch_size=128)
    
    source_patches = extract_patches_from_coords(source_img, coords)
    current_patches = extract_patches_from_coords(img, coords)

    clip_loss = calculate_directional_loss_patch(current_patches, source_patches, text_dir, model)

    bezier_pts_3d = bspline.to_bezier(bspline.points_3d)
    s_pos, s_width = bspline.get_jerk_loss()
    
    curve_area = img[..., 3] 
    
    # 2. Add the Batch and Channel dimensions to match objmask. Shape becomes [1, 1, H, W]
    curve_area = curve_area.unsqueeze(0).unsqueeze(0)
    mmse = multi_resolution_mse_loss(curve_area, objmask*0.6)
    manifold_loss = simple_mesh_fidelity_loss(bezier_pts_3d, mesh_vertices)
    loss =  0.5*mmse  + clip_loss  # + 0.001 * s_pos + 0.0001 *s_width # + 
    loss.backward()
    opt_bspline.step()
    opt_fixed.step()

    if step % 20 == 0:
        print(f"Step {step} | Manifold: {mmse.item():.4f} | Clip: {s_pos.item():.4f}| Smooth: {s_width.item():.4f}")
        pydiffvg.imwrite(img.cpu(), f"fill_steps/step_{step:03d}.png")
state = bspline.state_dict()
torch.save(state, "fill_steps/model_final.pth")
render_360_view(bspline)
for i, cam in enumerate(critical_cams):
    mesh_depth = render_native_depth_map(mesh_vertices, cam, 512, 512)
    scene_args = bspline.get_serialized_scene(cam, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth)
    img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
    # Save frame
    pydiffvg.imwrite(img.cpu(), f"fill_steps/final_{i:03d}.png")

Step 0 | Manifold: 0.7455 | Clip: 90.7327| Smooth: -0.0001
Step 20 | Manifold: 0.6389 | Clip: 102.4444| Smooth: 3432.6877
Step 40 | Manifold: 0.7724 | Clip: 112.8442| Smooth: 4536.1187
Step 60 | Manifold: 0.6268 | Clip: 121.7752| Smooth: 4831.0635
Step 80 | Manifold: 0.5864 | Clip: 128.6700| Smooth: 4912.2666
Step 100 | Manifold: 0.7388 | Clip: 137.4728| Smooth: 4958.3779
Step 120 | Manifold: 0.7464 | Clip: 145.6025| Smooth: 4973.3838
Step 140 | Manifold: 0.7105 | Clip: 153.2394| Smooth: 4998.0977
Step 160 | Manifold: 0.6276 | Clip: 160.1041| Smooth: 5005.5850
Step 180 | Manifold: 0.6946 | Clip: 167.1040| Smooth: 5006.6680
Step 200 | Manifold: 0.5976 | Clip: 173.8012| Smooth: 5039.6582
Step 220 | Manifold: 0.7328 | Clip: 181.3113| Smooth: 5019.5586
Step 240 | Manifold: 0.6915 | Clip: 187.6596| Smooth: 5016.4844


In [81]:
for i, cam in enumerate(critical_cams):
    mesh_depth = render_native_depth_map(mesh_vertices, cam, 512, 512)
    scene_args = bspline.get_serialized_scene(cam, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth)
    img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
    # Save frame
    pydiffvg.imwrite(img.cpu(), f"fill_steps/final_{i:03d}.png")

In [79]:
render_360_view(bspline)

/home/tobi/anaconda3/envs/diffvg/lib/python3.10/site-packages/diffvg-0.0.1-py3.10-linux-x86_64.egg/pydiffvg/render_pytorch.py:136: Warning: Detected non-closed paths with fill color. This might causes unexpected results.
  warnings.warn("Detected non-closed paths with fill color. This might causes unexpected results.", Warning)


GIF saved to fill_steps/optimization_360.gif


# Image Style

In [206]:
from PIL import Image
import torchvision.transforms as T

def get_image_style_direction(model, preprocess, target_style_path, source_style_path, device):
    """
    Computes the normalized semantic direction vector between two style images.
    target_style_path: Path to the masterpiece/reference (e.g., 'starry_night.jpg')
    source_style_path: Path to a neutral version (e.g., 'plain_sketch.png' or 'white_square.png')
    """
    # 1. Load and preprocess images using CLIP's native transform
    target_img = preprocess(Image.open(target_style_path)).unsqueeze(0).to(device)
    source_img = preprocess(Image.open(source_style_path)).unsqueeze(0).to(device)

    with torch.no_grad():
        # 2. Encode images to CLIP latent space
        target_features = model.encode_image(target_img)
        source_features = model.encode_image(source_img)

        # 3. Normalize
        target_features /= target_features.norm(dim=-1, keepdim=True)
        source_features /= source_features.norm(dim=-1, keepdim=True)

        # 4. Compute Direction (Target - Source)
        direction = target_features - source_features
        direction /= direction.norm(dim=-1, keepdim=True)

    return direction

style_dir = get_image_style_direction(
    model,
    preprocess_train,
    "stroke9.jpg",
    "gray.jpg",
    "cuda"
)

In [207]:
canvas_width = 512
canvas_height = 512
device = 'cuda' if torch.cuda.is_available() else 'cpu'

bspline = UnitaryBSpline3D_New(num_kp=target_points, 
                       kp_init=keypoints, 
                       device='cuda')

opt_bspline = torch.optim.Adam([bspline.points_3d], lr=0.003)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt_bspline, T_max=iter)

# Group 2: Other parameters (to stay constant)
opt_fixed = torch.optim.Adam([bspline.unit_widths], lr=0.2)

In [208]:
# --- Loop ---
for step in range(251):
    # if step == 100:
    #     print("\n>>> Triggering Adaptive Resampling: Doubling Resolution")
    #     redistribute_capacity(bspline, opt_bspline, opt_fixed)
    opt_bspline.zero_grad()
    opt_fixed.zero_grad()
    
    cam = get_perturbed_critical_view(device, perturbation_deg=10.0)
    # 1. Get the "Intrinsic" widths (the optimized thickness of the wire)
    # This remains consistent across all views
    mesh_depth = render_native_depth_map(mesh_vertices, cam, 512, 512)
    objmask = render_silhouette_mask(mesh_vertices, cam, view_w=512, view_h=512, splat_size=7)
    scene_args = bspline.get_serialized_scene(cam, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth)
    img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
    
    coords = get_jittered_coords(num_grid=4, patch_size=128)
    
    source_patches = extract_patches_from_coords(source_img, coords)
    current_patches = extract_patches_from_coords(img, coords)

    clip_loss = calculate_directional_loss_patch(current_patches, source_patches, style_dir, model)
    #manifold_loss = simple_mesh_fidelity_loss(bezier_pts_3d, mesh_vertices)
    bezier_pts_3d = bspline.to_bezier(bspline.points_3d)

    s_pos, s_width = bspline.get_jerk_loss()
    
    curve_area = img[..., 3] 
    
    # 2. Add the Batch and Channel dimensions to match objmask. Shape becomes [1, 1, H, W]
    curve_area = curve_area.unsqueeze(0).unsqueeze(0)
    mmse = multi_resolution_mse_loss(curve_area, objmask*0.8)

    loss =  0.2*mmse + clip_loss # + 0.001 * s_pos + 0.0001 *s_width# + 2*manifold_loss
    loss.backward()
    opt_bspline.step()
    opt_fixed.step()

    if step % 20 == 0:
        print(f"Step {step} | Manifold: {mmse.item():.4f} | Clip: {s_pos.item():.4f}| Smooth: {s_width.item():.4f}")
        pydiffvg.imwrite(img.cpu(), f"fill_steps/step_{step:03d}.png")
state = bspline.state_dict()
torch.save(state, "fill_steps/model_final.pth")
render_360_view(bspline)
for i, cam in enumerate(critical_cams):
    mesh_depth = render_native_depth_map(mesh_vertices, cam, 512, 512)
    scene_args = bspline.get_serialized_scene(cam, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth)
    img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
    # Save frame
    pydiffvg.imwrite(img.cpu(), f"fill_steps/final_{i:03d}.png")

Step 0 | Manifold: 1.5831 | Clip: 90.7327| Smooth: -0.0001
Step 20 | Manifold: 1.3296 | Clip: 29.3536| Smooth: 47.0642
Step 40 | Manifold: 1.2935 | Clip: 21.5775| Smooth: 13.8011
Step 60 | Manifold: 0.9414 | Clip: 19.4635| Smooth: 7.8782
Step 80 | Manifold: 1.2208 | Clip: 18.8212| Smooth: 5.6116
Step 100 | Manifold: 1.2493 | Clip: 18.7845| Smooth: 6.7191
Step 120 | Manifold: 1.1968 | Clip: 18.7313| Smooth: 5.3647
Step 140 | Manifold: 0.8873 | Clip: 18.8974| Smooth: 4.5209
Step 160 | Manifold: 1.0185 | Clip: 19.1078| Smooth: 4.5601
Step 180 | Manifold: 1.1767 | Clip: 19.7232| Smooth: 4.9380
Step 200 | Manifold: 1.1880 | Clip: 20.2754| Smooth: 4.9439
Step 220 | Manifold: 1.1785 | Clip: 20.6914| Smooth: 4.8092
Step 240 | Manifold: 1.1409 | Clip: 20.9727| Smooth: 4.6463


In [188]:
for i, cam in enumerate(critical_cams):
    mesh_depth = render_native_depth_map(mesh_vertices, cam, 512, 512)
    scene_args = bspline.get_serialized_scene(cam, canvas_width, canvas_height, mesh_vertices, normal, mesh_depth)
    img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
    # Save frame
    pydiffvg.imwrite(img.cpu(), f"fill_steps/final_{i:03d}.png")

In [85]:
for i, cam in enumerate(critical_cams):
    scene_args = bspline.get_serialized_scene(cam, canvas_width, canvas_height)
    img = pydiffvg.RenderFunction.apply(canvas_width, canvas_height, 2, 2, 0, None, *scene_args)
    # Save frame
    pydiffvg.imwrite(img.cpu(), f"fill_steps/final_{i:03d}.png")